<a href="https://colab.research.google.com/github/Abhishek1234563/-Robust-Multistatic-Target-Localization-/blob/main/MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Use CPU and float64 for numerical consistency
device = torch.device("cpu")
torch.set_default_dtype(torch.float64)

# Problem dimensions
d = 3
sensors = 5
samples = 200
p = sensors
n = samples

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.11.0+cpu
Device: cpu


In [ ]:
# Transmitter position
T = np.array([30.0, -15.0, 40.0])

# Initial target estimate
u0 = np.array([-5.0, 5.0, 10.0])

# Measurement noise
noise_db = 0
N_pow = 10.0 ** (noise_db / 10)

# Outlier settings for the initial simulation
PER = 0.20
outlier_magnitude = 505.0

# Generate receiver positions
r1 = np.random.randn(d, p)

rr = r1 / np.linalg.norm(
    r1,
    axis=0,
    keepdims=True
)

# Original receiver layout: shape (3, 5)
r = rr * 50.0 + u0[:, None]

# Convert geometry to PyTorch tensors.
# Receiver tensor uses shape (5, 3): one receiver per row.
T_torch = torch.tensor(
    T,
    dtype=torch.float64,
    device=device
)

r_torch = torch.tensor(
    r.T,
    dtype=torch.float64,
    device=device
)

print("T shape:", T_torch.shape)
print("r_torch shape:", r_torch.shape)

T shape: torch.Size([3])
r_torch shape: torch.Size([5, 3])


In [ ]:
def bistatic_ranges(x, T, r):
    """
    Calculate bistatic ranges from transmitter to target
    and target to each receiver.

    x: target position, shape (3,)
    T: transmitter position, shape (3,)
    r: receiver positions, shape (3, 5)
    """

    tx_dist = np.linalg.norm(x - T)

    rx_dist = np.linalg.norm(
        x[:, None] - r,
        axis=0
    )

    return tx_dist + rx_dist

In [ ]:
def generate_measurements():
    # Measurements at the initial target position
    Y0 = bistatic_ranges(u0, T, r)

    # Add Gaussian noise
    Y = (
        Y0[:, None]
        + np.random.randn(p, n) * np.sqrt(N_pow)
    )

    Y_clean = Y.copy()

    # Track injected outliers
    true_outliers = np.zeros(
        (p, n),
        dtype=bool
    )

    N_c = int(np.floor(PER * n))

    for i in range(p):
        indices = np.random.permutation(n)[:N_c]

        half = N_c // 2

        plus_indices = indices[:half]
        minus_indices = indices[half:]

        Y[i, plus_indices] += outlier_magnitude
        Y[i, minus_indices] -= outlier_magnitude

        true_outliers[i, indices] = True

    return Y, Y_clean, true_outliers

In [ ]:
def generate_training_scenario():
    # Random true target position
    x_true = np.random.uniform(
        -20.0,
        20.0,
        size=3
    )

    # True bistatic ranges
    tx_dist = np.linalg.norm(x_true - T)

    rx_dist = np.linalg.norm(
        x_true[:, None] - r,
        axis=0
    )

    true_ranges = tx_dist + rx_dist

    # Random noise standard deviation
    noise_std = np.random.uniform(0.1, 2.0)

    noise = np.random.randn(p, n) * noise_std

    Y = true_ranges[:, None] + noise

    # Label 1 = clean measurement
    # Label 0 = corrupted measurement
    labels = np.ones(
        (p, n),
        dtype=np.float64
    )

    # Random corruption percentage
    corruption_fraction = np.random.uniform(
        0.05,
        0.30
    )

    num_corruptions = int(
        corruption_fraction * n
    )

    for i in range(p):
        indices = np.random.choice(
            n,
            size=num_corruptions,
            replace=False
        )

        magnitudes = np.random.uniform(
            2.0,
            20.0,
            size=num_corruptions
        )

        signs = np.random.choice(
            [-1.0, 1.0],
            size=num_corruptions
        )

        Y[i, indices] += magnitudes * signs

        labels[i, indices] = 0.0

    # Convert to PyTorch tensors
    Y_tensor = torch.tensor(
        Y,
        dtype=torch.float64,
        device=device
    )

    x_true_tensor = torch.tensor(
        x_true,
        dtype=torch.float64,
        device=device
    )

    labels_tensor = torch.tensor(
        labels,
        dtype=torch.float64,
        device=device
    )

    return (
        Y_tensor,
        x_true_tensor,
        labels_tensor
    )

In [ ]:
num_train_scenarios = 64
num_val_scenarios = 16

train_data = [
    generate_training_scenario()
    for _ in range(num_train_scenarios)
]

val_data = [
    generate_training_scenario()
    for _ in range(num_val_scenarios)
]

print("Training scenarios:", len(train_data))
print("Validation scenarios:", len(val_data))

print("Measurement shape:", train_data[0][0].shape)
print("Target shape:", train_data[0][1].shape)
print("Label shape:", train_data[0][2].shape)

Training scenarios: 64
Validation scenarios: 16
Measurement shape: torch.Size([5, 200])
Target shape: torch.Size([3])
Label shape: torch.Size([5, 200])


In [ ]:
def calculate_features(x, Y, T, r, residual_scale):
    # Predicted bistatic ranges
    tx_dist = torch.linalg.norm(x - T)

    rx_dist = torch.linalg.norm(
        x[None, :] - r,
        dim=1
    )

    predicted_ranges = tx_dist + rx_dist[:, None]

    # Residuals
    residuals = Y - predicted_ranges

    # Normalized residuals
    z = residuals / residual_scale

    # Receiver-level statistics
    receiver_mean = torch.mean(
        z,
        dim=1,
        keepdim=True
    )

    receiver_rms = torch.sqrt(
        torch.mean(
            z ** 2,
            dim=1,
            keepdim=True
        ) + 1e-12
    )

    # Scene-level statistics
    scene_mean = torch.mean(z)

    scene_rms = torch.sqrt(
        torch.mean(z ** 2) + 1e-12
    )

    # Six features per measurement
    features = torch.stack([
        z,
        torch.abs(z),
        receiver_mean.expand_as(z),
        receiver_rms.expand_as(z),
        scene_mean.expand_as(z),
        scene_rms.expand_as(z)
    ], dim=-1)

    return features

In [ ]:
def weighted_position_update(x, Y, weights, T, r):
    eps = 1e-12

    # Distances
    D_A = torch.linalg.norm(
        x - T
    ).clamp_min(eps)

    D_B = torch.linalg.norm(
        x[None, :] - r,
        dim=1
    ).clamp_min(eps)

    # Unit direction vectors
    u = (x - T) / D_A

    v = (
        x[None, :] - r
    ) / D_B[:, None]

    # Distance ratios
    h = D_A / D_B
    g = D_B / D_A

    # Numerator terms
    c2 = Y[:, :, None] * (
        u[None, None, :]
        + v[:, None, :]
    )

    c3 = (
        (1.0 + g)[:, None, None]
        * T
    )

    c4 = (
        (1.0 + h)[:, None, None]
        * r[:, None, :]
    )

    numerator_terms = c2 + c3 + c4

    # Denominator terms
    denominator_terms = (
        2.0 + h + g
    )[:, None]

    numerator = torch.sum(
        weights[:, :, None] * numerator_terms,
        dim=(0, 1)
    )

    denominator = torch.sum(
        weights * denominator_terms
    )

    x_next = numerator / denominator.clamp_min(eps)

    return x_next

In [ ]:
class WeightMLP(nn.Module):
    def __init__(self, input_features=6):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 16),
            nn.Tanh(),

            nn.Linear(16, 16),
            nn.Tanh(),

            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)

In [ ]:
class UnrolledROTALO(nn.Module):
    def __init__(
        self,
        stages=10,
        weight_floor=0.001
    ):
        super().__init__()

        self.stages = stages
        self.weight_floor = weight_floor

        self.weight_net = WeightMLP(
            input_features=6
        )

    def forward(
        self,
        x_initial,
        Y,
        T,
        r,
        residual_scale
    ):
        x = x_initial

        all_scores = []
        all_weights = []

        for _ in range(self.stages):
            # Calculate measurement features
            features = calculate_features(
                x,
                Y,
                T,
                r,
                residual_scale
            )

            # Flatten measurements for MLP
            features_flat = features.reshape(
                -1,
                6
            )

            # MLP scores
            scores = self.weight_net(
                features_flat
            ).reshape(
                Y.shape[0],
                Y.shape[1]
            )

            # Apply weight floor
            weights = (
                self.weight_floor
                + (1.0 - self.weight_floor) * scores
            )

            # Update target position
            x = weighted_position_update(
                x,
                Y,
                weights,
                T,
                r
            )

            all_scores.append(scores)
            all_weights.append(weights)

        return x, all_scores, all_weights

In [ ]:
lambda_cls = 0.1

def calculate_losses(
    x_pred,
    x_true,
    scores,
    labels
):
    # Normalized position loss
    position_loss = torch.mean(
        ((x_pred - x_true) / 50.0) ** 2
    )

    # Clean/corrupted classification loss
    classification_loss = (
        nn.functional.binary_cross_entropy(
            scores,
            labels
        )
    )

    # Combined loss
    total_loss = (
        position_loss
        + lambda_cls * classification_loss
    )

    return (
        total_loss,
        position_loss,
        classification_loss
    )

In [ ]:
model = UnrolledROTALO(
    stages=10,
    weight_floor=0.001
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Model initialized.")

Model initialized.


In [ ]:
def train_one_scenario(scenario):
    Y, x_true, labels = scenario

    # Fixed operational initial estimate
    x_initial = torch.tensor(
        u0,
        dtype=torch.float64,
        device=device
    )

    # Calculate initial residual scale
    with torch.no_grad():
        tx_dist = torch.linalg.norm(
            x_initial - T_torch
        )

        rx_dist = torch.linalg.norm(
            x_initial[None, :] - r_torch,
            dim=1
        )

        predicted_initial = (
            tx_dist + rx_dist[:, None]
        )

        initial_residuals = (
            Y - predicted_initial
        )

        residual_scale = torch.sqrt(
            torch.mean(
                initial_residuals ** 2
            )
        ).clamp_min(0.05)

    # Forward pass
    x_pred, all_scores, all_weights = model(
        x_initial,
        Y,
        T_torch,
        r_torch,
        residual_scale
    )

    # Final-stage scores
    final_scores = all_scores[-1]

    # Calculate loss
    total_loss, position_loss, classification_loss = (
        calculate_losses(
            x_pred,
            x_true,
            final_scores,
            labels
        )
    )

    # Backpropagation
    optimizer.zero_grad()

    total_loss.backward()

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    return (
        total_loss.item(),
        position_loss.item(),
        classification_loss.item(),
        x_pred.detach()
    )

In [ ]:
loss, pos_loss, cls_loss, x_pred = train_one_scenario(
    train_data[0]
)

print("Total loss:", loss)
print("Position loss:", pos_loss)
print("Classification loss:", cls_loss)
print("Predicted position:", x_pred.numpy())

print(
    "All losses finite:",
    np.isfinite([
        loss,
        pos_loss,
        cls_loss
    ]).all()
)

Total loss: 0.09239737390441537
Position loss: 0.019288207336720858
Classification loss: 0.7310916656769451
Predicted position: [-2.91241218 -8.98319857 -0.5335149 ]
All losses finite: True


In [ ]:
def evaluate_model(data):
    total_losses = []
    position_losses = []
    classification_losses = []

    model.eval()

    with torch.no_grad():
        for scenario in data:
            Y, x_true, labels = scenario

            # Initial estimate
            x_initial = torch.tensor(
                u0,
                dtype=torch.float64,
                device=device
            )

            # Initial residual scale
            tx_dist = torch.linalg.norm(
                x_initial - T_torch
            )

            rx_dist = torch.linalg.norm(
                x_initial[None, :] - r_torch,
                dim=1
            )

            predicted_initial = (
                tx_dist + rx_dist[:, None]
            )

            initial_residuals = (
                Y - predicted_initial
            )

            residual_scale = torch.sqrt(
                torch.mean(
                    initial_residuals ** 2
                )
            ).clamp_min(0.05)

            # Forward pass
            x_pred, all_scores, all_weights = model(
                x_initial,
                Y,
                T_torch,
                r_torch,
                residual_scale
            )

            # Loss
            total_loss, position_loss, classification_loss = (
                calculate_losses(
                    x_pred,
                    x_true,
                    all_scores[-1],
                    labels
                )
            )

            total_losses.append(
                total_loss.item()
            )

            position_losses.append(
                position_loss.item()
            )

            classification_losses.append(
                classification_loss.item()
            )

    model.train()

    return (
        np.mean(total_losses),
        np.mean(position_losses),
        np.mean(classification_losses)
    )

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()

    train_total_losses = []
    train_position_losses = []
    train_classification_losses = []

    # Shuffle training scenarios
    indices = np.random.permutation(
        len(train_data)
    )

    for idx in indices:
        loss, pos_loss, cls_loss, x_pred = (
            train_one_scenario(
                train_data[idx]
            )
        )

        train_total_losses.append(loss)
        train_position_losses.append(pos_loss)
        train_classification_losses.append(cls_loss)

    # Validation
    val_total, val_pos, val_cls = evaluate_model(
        val_data
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs}"
    )

    print(
        f"Train Total Loss: "
        f"{np.mean(train_total_losses):.6f}"
    )

    print(
        f"Train Position Loss: "
        f"{np.mean(train_position_losses):.6f}"
    )

    print(
        f"Train Classification Loss: "
        f"{np.mean(train_classification_losses):.6f}"
    )

    print(
        f"Validation Total Loss: "
        f"{val_total:.6f}"
    )

    print(
        f"Validation Position Loss: "
        f"{val_pos:.6f}"
    )

    print(
        f"Validation Classification Loss: "
        f"{val_cls:.6f}"
    )

    print("-" * 50)

Epoch 1/5
Train Total Loss: 0.093982
Train Position Loss: 0.030246
Train Classification Loss: 0.637362
Validation Total Loss: 0.080610
Validation Position Loss: 0.027172
Validation Classification Loss: 0.534380
--------------------------------------------------
Epoch 2/5
Train Total Loss: 0.079646
Train Position Loss: 0.030196
Train Classification Loss: 0.494500
Validation Total Loss: 0.073379
Validation Position Loss: 0.027399
Validation Classification Loss: 0.459804
--------------------------------------------------
Epoch 3/5
Train Total Loss: 0.075299
Train Position Loss: 0.030483
Train Classification Loss: 0.448156
Validation Total Loss: 0.070305
Validation Position Loss: 0.027912
Validation Classification Loss: 0.423929
--------------------------------------------------
Epoch 4/5
Train Total Loss: 0.071926
Train Position Loss: 0.031070
Train Classification Loss: 0.408556
Validation Total Loss: 0.067324
Validation Position Loss: 0.028703
Validation Classification Loss: 0.386215
---

In [ ]:
torch.save(
    model.state_dict(),
    "unrolled_rotalo_pilot.pth"
)

print("Model saved.")

Model saved.


In [ ]:
# Check geometry tensor shapes
print("T_torch:", T_torch.shape)
print("r_torch:", r_torch.shape)

# Check one training scenario
Y, x_true, labels = train_data[0]

print("Y:", Y.shape)
print("x_true:", x_true.shape)
print("labels:", labels.shape)

# Check feature calculation
x_initial = torch.tensor(
    u0,
    dtype=torch.float64,
    device=device
)

with torch.no_grad():
    tx_dist = torch.linalg.norm(x_initial - T_torch)

    rx_dist = torch.linalg.norm(
        x_initial[None, :] - r_torch,
        dim=1
    )

    predicted_initial = tx_dist + rx_dist[:, None]

    initial_residuals = Y - predicted_initial

    residual_scale = torch.sqrt(
        torch.mean(initial_residuals ** 2)
    ).clamp_min(0.05)

    features = calculate_features(
        x_initial,
        Y,
        T_torch,
        r_torch,
        residual_scale
    )

print("Residual scale:", residual_scale.item())
print("Feature shape:", features.shape)
print("Features finite:", torch.isfinite(features).all().item())

T_torch: torch.Size([3])
r_torch: torch.Size([5, 3])
Y: torch.Size([5, 200])
x_true: torch.Size([3])
labels: torch.Size([5, 200])
Residual scale: 18.810498494201656
Feature shape: torch.Size([5, 200, 6])
Features finite: True


In [ ]:
loss, pos_loss, cls_loss, x_pred = train_one_scenario(
    train_data[0]
)

print("Total loss:", loss)
print("Position loss:", pos_loss)
print("Classification loss:", cls_loss)
print("Predicted position:", x_pred.numpy())

print(
    "All losses finite:",
    np.isfinite([loss, pos_loss, cls_loss]).all()
)

Total loss: 0.0777788607710142
Position loss: 0.02181234000791854
Classification loss: 0.5596652076309565
Predicted position: [-3.03236786 -8.30751162  0.04519603]
All losses finite: True


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()

    train_total_losses = []
    train_position_losses = []
    train_classification_losses = []

    # Shuffle training scenarios
    indices = np.random.permutation(len(train_data))

    for idx in indices:
        loss, pos_loss, cls_loss, x_pred = train_one_scenario(
            train_data[idx]
        )

        train_total_losses.append(loss)
        train_position_losses.append(pos_loss)
        train_classification_losses.append(cls_loss)

    # Evaluate on the separate validation set
    val_total, val_pos, val_cls = evaluate_model(val_data)

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Total Loss: {np.mean(train_total_losses):.6f}")
    print(f"Train Position Loss: {np.mean(train_position_losses):.6f}")
    print(f"Train Classification Loss: {np.mean(train_classification_losses):.6f}")
    print(f"Validation Total Loss: {val_total:.6f}")
    print(f"Validation Position Loss: {val_pos:.6f}")
    print(f"Validation Classification Loss: {val_cls:.6f}")
    print("-" * 50)

Epoch 1/5
Train Total Loss: 0.067412
Train Position Loss: 0.032062
Train Classification Loss: 0.353505
Validation Total Loss: 0.063664
Validation Position Loss: 0.029434
Validation Classification Loss: 0.342295
--------------------------------------------------
Epoch 2/5
Train Total Loss: 0.066058
Train Position Loss: 0.032056
Train Classification Loss: 0.340024
Validation Total Loss: 0.062471
Validation Position Loss: 0.029294
Validation Classification Loss: 0.331771
--------------------------------------------------
Epoch 3/5
Train Total Loss: 0.064905
Train Position Loss: 0.032113
Train Classification Loss: 0.327923
Validation Total Loss: 0.061639
Validation Position Loss: 0.029406
Validation Classification Loss: 0.322334
--------------------------------------------------
Epoch 4/5
Train Total Loss: 0.063915
Train Position Loss: 0.031918
Train Classification Loss: 0.319966
Validation Total Loss: 0.060400
Validation Position Loss: 0.029094
Validation Classification Loss: 0.313062
---

In [ ]:
def evaluate_position_rmse(data):
    squared_errors = []

    model.eval()

    with torch.no_grad():
        for scenario in data:
            Y, x_true, labels = scenario

            # Fixed initial estimate
            x_initial = torch.tensor(
                u0,
                dtype=torch.float64,
                device=device
            )

            # Initial residual scale
            tx_dist = torch.linalg.norm(
                x_initial - T_torch
            )

            rx_dist = torch.linalg.norm(
                x_initial[None, :] - r_torch,
                dim=1
            )

            predicted_initial = (
                tx_dist + rx_dist[:, None]
            )

            initial_residuals = (
                Y - predicted_initial
            )

            residual_scale = torch.sqrt(
                torch.mean(
                    initial_residuals ** 2
                )
            ).clamp_min(0.05)

            # Run trained model
            x_pred, _, _ = model(
                x_initial,
                Y,
                T_torch,
                r_torch,
                residual_scale
            )

            # Squared position error
            squared_error = torch.sum(
                (x_pred - x_true) ** 2
            )

            squared_errors.append(
                squared_error.item()
            )

    model.train()

    rmse = np.sqrt(
        np.mean(squared_errors)
    )

    return rmse


val_rmse = evaluate_position_rmse(val_data)

print("Validation Position RMSE:", val_rmse, "meters")

Validation Position RMSE: 14.716583006933305 meters
